In [16]:
import geopandas as gpd
import pandas as pd
import geoplanar
import numpy as np
from shapely.ops import unary_union
import shapely


source: https://geoportaal.maaamet.ee/eng/spatial-data/estonian-topographic-database/download-topographic-data-p618.html

In [51]:
gdf = gpd.read_file("/data/uscuni-ulce/extension2/est/est_shp/E_401_hoone_ka.shp")

In [52]:
gdf

,etak_id,kood,kood_t,tyyp,tyyp_t,ehr_gid,ads_oid,ads_lahiaa,kov_id,markused,korgus_m,vajalik,vajalik_t,andmeallik,korgusalli,ruumikujua,muutmisaeg,geom_muutm,valjavote,geometry
0,342474,401,Hoone,30,Vundament,NaN,NaN,NaN,NaN,NaN,NaN,30,Korras,216,216,216,2019-09-04,2011-03-01,2026-08-08,"POLYGON Z ((685243.534 6497620.568 35.588, 685..."
1,267523,401,Hoone,10,Elu- või ühiskondlik hoone,108011215,EE00919896,Tammiku tn 18,NaN,NaN,8.0,30,Korras,210,214,214,2020-05-14,2018-08-29,2026-08-08,"POLYGON Z ((633703.036 6579288.484 90.822, 633..."
2,299352,401,Hoone,10,Elu- või ühiskondlik hoone,108001241,EE00889742,Urvaste,NaN,NaN,NaN,30,Korras,152,210,152,2018-03-03,2014-05-21,2026-08-08,"POLYGON Z ((635480.513 6591442.435 63.609, 635..."
3,280187,401,Hoone,10,Elu- või ühiskondlik hoone,108001871,EE00892007,Jõe tn 5,NaN,NaN,7.0,30,Korras,221,226,226,2024-01-27,2022-09-14,2026-08-08,"POLYGON Z ((621723.276 6579521.937 91.006, 621..."
4,265614,401,Hoone,10,Elu- või ühiskondlik hoone,108031302,EE00832882,Pargi tn 10,NaN,NaN,7.0,30,Korras,221,210,152,2024-01-27,2014-04-29,2026-08-08,"POLYGON Z ((621626.396 6579915.88 98.101, 6216..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
924452,9812088,401,Hoone,10,Elu- või ühiskondlik hoone,NaN,NaN,NaN,NaN,NaN,NaN,30,Korras,235,235,235,2026-08-04,2026-08-04,2026-08-08,"POLYGON Z ((514078.954 6459635.226 7.041, 5140..."
924453,9812021,401,Hoone,30,Vundament,NaN,NaN,NaN,NaN,NaN,NaN,30,Korras,235,235,235,2026-08-04,2026-08-04,2026-08-08,"POLYGON Z ((516053.426 6462104.512 4.913, 5160..."
924454,9811965,401,Hoone,30,Vundament,NaN,NaN,NaN,NaN,NaN,NaN,30,Korras,235,235,235,2026-08-04,2026-08-04,2026-08-08,"POLYGON Z ((549019.24 6477079.401 18.384, 5490..."
924455,9811967,401,Hoone,30,Vundament,NaN,NaN,NaN,NaN,NaN,NaN,30,Korras,235,235,235,2026-08-04,2026-08-04,2026-08-08,"POLYGON Z ((549080.495 6477098.773 18.125, 549..."


Check for weirdly large polygons

In [53]:
gdf["area"] = gdf.geometry.area
largest_100 = gdf.nlargest(100, "area")

In [ ]:
largest_100.explore()

check for overlaps

In [54]:
pairs = geoplanar.overlaps(gdf)
overlap_idx = np.unique(pairs)

In [55]:
gdf_overlaps = gdf.iloc[overlap_idx]

In [ ]:
gdf_overlaps.explore()

In [56]:
gdf = geoplanar.trim_overlaps(gdf, strategy='smallest')

Age of the building might be useful at some point

In [57]:
gdf_clean = gdf[["geometry"]].copy()

In [58]:
gdf_clean = gdf_clean.explode(index_parts=False).reset_index(drop=True)

In [60]:
gdf_clean = gdf_clean.to_crs(epsg=3035)

In [61]:
gdf_clean

,geometry
0,"POLYGON Z ((5311192.588 4062247.032 35.588, 53..."
1,"POLYGON Z ((5244054.206 4132394.211 90.822, 52..."
2,"POLYGON Z ((5243298.343 4144659.396 63.609, 52..."
3,"POLYGON Z ((5232278.875 4130291.133 91.006, 52..."
4,"POLYGON Z ((5232103.17 4130658.589 98.101, 523..."
...,...
924459,"POLYGON Z ((5151384.378 3991710.115 7.041, 515..."
924460,"POLYGON Z ((5152815.177 3994517.916 4.913, 515..."
924461,"POLYGON Z ((5182041.993 4015641.251 18.384, 51..."
924462,"POLYGON Z ((5182098.017 4015672.199 18.125, 51..."


In [63]:
gdf_clean.to_parquet("/data/uscuni-ulce/extension2/est/clean_3035.parquet")